# ML Model Registry — Tracking LUMI EnergyHub Forecasting Models

## Objective

This notebook registers the forecasting models trained in `DOE_arima_forecasting.ipynb` into the Supabase `ml_model_registry` table. It demonstrates:

1. Loading model comparison results from the ARIMA notebook
2. Connecting to Supabase via `supabase-py`
3. Inserting model metadata (name, version, type, metrics, path)
4. Activating the best-performing model per target variable
5. Querying the registry for dashboards and reproducibility

> **Note:** The `ml_model_registry` table schema supports ARIMA, SARIMA, LightGBM, XGBoost, and Prophet. Our thesis scope uses SARIMA (which includes ARIMA as a subset).

## Step 1: Setup — Load Environment & Libraries

In [1]:
import os
import json
from datetime import datetime, date
from pathlib import Path

import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Supabase client
from supabase import create_client, Client

# Load .env from project root (two dirs up from this notebook)
notebook_dir = Path.cwd()
env_path = notebook_dir.parent / '.env'
load_dotenv(env_path)

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_SERVICE_KEY = os.getenv('SUPABASE_JWT_SERVICE_ROLE_KEY') or os.getenv('SUPABASE_SERVICE_ROLE_KEY')

if not SUPABASE_URL or not SUPABASE_SERVICE_KEY:
    raise ValueError("Missing SUPABASE_URL or SUPABASE_SERVICE_ROLE_KEY in .env")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
print('Supabase client connected.')
print('Project:', SUPABASE_URL.split('//')[-1].split('.')[0])

Supabase client connected.
Project: husnkzlccdrjpwlqcfbt


## Step 2: Load Model Comparison Results

These results were generated by `DOE_arima_forecasting.ipynb` and saved as `model_comparison_results.csv`.

In [2]:
# Load the model comparison results
results_path = notebook_dir / 'model_comparison_results.csv'
if not results_path.exists():
    raise FileNotFoundError(f'Run DOE_arima_forecasting.ipynb first to generate {results_path}')

results_df = pd.read_csv(results_path)
print('=== Model Comparison Results ===')
print(results_df.to_string(index=False))

# Identify the best model (lowest MAE)
best_model = results_df.loc[results_df['mae'].idxmin()]
print(f"\nBest model: {best_model['model']} | MAE={best_model['mae']:.2f} | MAPE={best_model['mape']:.2f}%")

=== Model Comparison Results ===
                   model          mae         rmse      mape
 Linear Trend Regression  5993.830753  7342.095036  4.965126
   Holt Linear Smoothing  6557.716104  7997.679152  5.435109
        Naive with Drift  6709.323529  8128.453097  5.566170
            ARIMA(1,1,1)  6829.090287  8257.124448  5.667112
   SARIMAX(1,1,1) + Exog  9913.651842 11459.109209  8.282299
Random Forest Regression 15957.407151 17806.288845 13.406408

Best model: Linear Trend Regression | MAE=5993.83 | MAPE=4.97%


## Step 3: Map Notebook Models to Registry Schema

The `ml_model_registry` table expects:
- `model_name` — human-readable identifier
- `model_version` — semantic version (e.g., "1.0.0")
- `model_type` — SARIMA, LightGBM, XGBoost, or Prophet
- `target_variable` — what we are predicting
- `train_date` — when the model was trained
- `metrics` — JSONB with MAE, RMSE, MAPE
- `model_path` — optional path to serialized model or CSV artifact
- `is_active` — whether this is the production model for its target

In [3]:
# Map each model from the comparison to registry format
def map_to_registry(row: pd.Series, target: str, model_path: str = None) -> dict:
    """Convert a model comparison row into ml_model_registry format."""
    model_name = row['model']
    
    # Determine model_type from name
    if 'ARIMA' in model_name or 'SARIMAX' in model_name or 'Holt' in model_name:
        model_type = 'SARIMA'  # SARIMA covers ARIMA, SARIMAX, and exponential smoothing in our taxonomy
    elif 'Random Forest' in model_name or 'Linear' in model_name:
        model_type = 'LightGBM'  # Tree-based / regression bucket for non-statistical models
    else:
        model_type = 'SARIMA'
    
    return {
        'model_name': f"{model_name} — {target}",
        'model_version': '1.0.0',  # First training run
        'model_type': model_type,
        'target_variable': target,
        'train_date': date.today().isoformat(),
        'metrics': {
            'mae': round(float(row['mae']), 4),
            'rmse': round(float(row['rmse']), 4),
            'mape': round(float(row['mape']), 4),
            'test_period': '2021-2024',
            'train_period': '2003-2020',
            'n_train': 18,
            'n_test': 4
        },
        'model_path': model_path or f"notebooks/DOE_arima_forecasting_{target}.csv",
        'is_active': False  # We'll activate the best one later
    }

# Build registry rows for total_consumption_gwh
consumption_models = [
    map_to_registry(row, target='total_consumption_gwh', model_path='forecast_consumption_2025_2030.csv')
    for _, row in results_df.iterrows()
]

print(f"Prepared {len(consumption_models)} models for registry insertion.")
print('Sample entry:')
print(json.dumps(consumption_models[0], indent=2))

Prepared 6 models for registry insertion.
Sample entry:
{
  "model_name": "Linear Trend Regression \u2014 total_consumption_gwh",
  "model_version": "1.0.0",
  "model_type": "LightGBM",
  "target_variable": "total_consumption_gwh",
  "train_date": "2026-06-11",
  "metrics": {
    "mae": 5993.8308,
    "rmse": 7342.095,
    "mape": 4.9651,
    "test_period": "2021-2024",
    "train_period": "2003-2020",
    "n_train": 18,
    "n_test": 4
  },
  "model_path": "forecast_consumption_2025_2030.csv",
  "is_active": false
}


## Step 4: Insert Models into Supabase Registry

We insert all 6 models for `total_consumption_gwh`. The unique partial index on `(target_variable, is_active)` ensures only one model per target can be active at a time.

In [4]:
def upsert_model(model: dict) -> dict:
    """Insert or update a model in ml_model_registry."""
    response = (
        supabase
        .table('ml_model_registry')
        .insert(model)
        .execute()
    )
    return response.data[0] if response.data else None

# Insert all consumption models
inserted = []
for m in consumption_models:
    try:
        record = upsert_model(m)
        inserted.append(record)
        print(f"Inserted: {record['model_name']} ({record['model_id']})")
    except Exception as e:
        print(f"Error inserting {m['model_name']}: {e}")

print(f"\nTotal inserted: {len(inserted)}")

Inserted: Linear Trend Regression — total_consumption_gwh (cd4be254-38ff-4e7f-b998-87fe62a94f8c)
Inserted: Holt Linear Smoothing — total_consumption_gwh (5527fc0c-9785-4f59-9967-a9e6690d7afd)
Inserted: Naive with Drift — total_consumption_gwh (5645bdaf-152a-4cd6-baad-d6c0c9d36517)
Inserted: ARIMA(1,1,1) — total_consumption_gwh (32894efc-0dad-4cf1-8338-e8d4e4bffb13)
Inserted: SARIMAX(1,1,1) + Exog — total_consumption_gwh (4335b279-467a-4f5a-9ead-8516be013e02)
Inserted: Random Forest Regression — total_consumption_gwh (07e6ffde-5e88-44ab-8c5c-c1f24f9040c0)

Total inserted: 6


## Step 5: Activate the Best Model

The partial unique index `idx_ml_model_active_unique` ensures only one `(target_variable, is_active=true)` pair exists. We deactivate all others, then activate the best.

In [5]:
def activate_model(model_id: str, target: str) -> dict:
    """
    Deactivate all models for this target, then activate the chosen one.
    The partial unique index handles the constraint at the DB level.
    """
    # Step 1: deactivate all models for this target
    supabase.table('ml_model_registry')\
        .update({'is_active': False})\
        .eq('target_variable', target)\
        .execute()
    
    # Step 2: activate the chosen model
    response = (
        supabase
        .table('ml_model_registry')
        .update({'is_active': True})
        .eq('model_id', model_id)
        .execute()
    )
    return response.data[0] if response.data else None

# Find the best model ID from our inserted records
best_id = None
for rec in inserted:
    if rec['model_name'].startswith(best_model['model']):
        best_id = rec['model_id']
        break

if best_id:
    activated = activate_model(best_id, 'total_consumption_gwh')
    print(f"Activated best model: {activated['model_name']}")
    print(f"  model_id: {activated['model_id']}")
    print(f"  metrics: {json.dumps(activated['metrics'], indent=2)}")
else:
    print('Best model ID not found. Check inserted records.')

Activated best model: Linear Trend Regression — total_consumption_gwh
  model_id: cd4be254-38ff-4e7f-b998-87fe62a94f8c
  metrics: {
  "mae": 5993.8308,
  "mape": 4.9651,
  "rmse": 7342.095,
  "n_test": 4,
  "n_train": 18,
  "test_period": "2021-2024",
  "train_period": "2003-2020"
}


## Step 6: Query the Registry

Demonstrate how the LUMI backend would query the registry at runtime to:
1. Find the active model for a target variable
2. List all models of a given type
3. Compare metrics across versions

In [6]:
# 6.1: Get the active model for total_consumption_gwh
active_resp = (
    supabase
    .table('ml_model_registry')
    .select('*')
    .eq('target_variable', 'total_consumption_gwh')
    .eq('is_active', True)
    .single()
    .execute()
)

if active_resp.data:
    print('=== Active Model for total_consumption_gwh ===')
    print(json.dumps(active_resp.data, indent=2, default=str))
else:
    print('No active model found.')

=== Active Model for total_consumption_gwh ===
{
  "model_id": "cd4be254-38ff-4e7f-b998-87fe62a94f8c",
  "model_name": "Linear Trend Regression \u2014 total_consumption_gwh",
  "model_version": "1.0.0",
  "model_type": "LightGBM",
  "target_variable": "total_consumption_gwh",
  "train_date": "2026-06-11",
  "metrics": {
    "mae": 5993.8308,
    "mape": 4.9651,
    "rmse": 7342.095,
    "n_test": 4,
    "n_train": 18,
    "test_period": "2021-2024",
    "train_period": "2003-2020"
  },
  "model_path": "forecast_consumption_2025_2030.csv",
  "is_active": true,
  "created_at": "2026-06-11T14:33:51.134076+00:00",
  "updated_at": "2026-06-11T14:33:52.808952+00:00"
}


In [7]:
# 6.2: List all SARIMA models ordered by test MAPE
sarima_resp = (
    supabase
    .table('ml_model_registry')
    .select('*')
    .eq('model_type', 'SARIMA')
    .eq('target_variable', 'total_consumption_gwh')
    .order('train_date', desc=True)
    .execute()
)

print(f"=== SARIMA Models for total_consumption_gwh ({len(sarima_resp.data)} found) ===")
for m in sarima_resp.data:
    metrics = m['metrics']
    print(f"  {m['model_name']}")
    print(f"    MAE={metrics['mae']:.2f}  RMSE={metrics['rmse']:.2f}  MAPE={metrics['mape']:.2f}%  active={m['is_active']}")

=== SARIMA Models for total_consumption_gwh (4 found) ===
  Holt Linear Smoothing — total_consumption_gwh
    MAE=6557.72  RMSE=7997.68  MAPE=5.44%  active=False
  Naive with Drift — total_consumption_gwh
    MAE=6709.32  RMSE=8128.45  MAPE=5.57%  active=False
  ARIMA(1,1,1) — total_consumption_gwh
    MAE=6829.09  RMSE=8257.12  MAPE=5.67%  active=False
  SARIMAX(1,1,1) + Exog — total_consumption_gwh
    MAE=9913.65  RMSE=11459.11  MAPE=8.28%  active=False


In [8]:
# 6.3: Full registry overview (all targets)
all_resp = (
    supabase
    .table('ml_model_registry')
    .select('*')
    .order('target_variable')
    .order('metrics->mape')
    .execute()
)

rows = []
for m in all_resp.data:
    rows.append({
        'target': m['target_variable'],
        'model': m['model_name'].split(' — ')[0],
        'type': m['model_type'],
        'version': m['model_version'],
        'mae': m['metrics']['mae'],
        'mape': m['metrics']['mape'],
        'active': m['is_active'],
        'trained': m['train_date']
    })

overview = pd.DataFrame(rows)
print('=== Full Model Registry Overview ===')
print(overview.to_string(index=False))

=== Full Model Registry Overview ===
               target                    model     type version        mae    mape  active    trained
total_consumption_gwh  Linear Trend Regression LightGBM   1.0.0  5993.8308  4.9651    True 2026-06-11
total_consumption_gwh    Holt Linear Smoothing   SARIMA   1.0.0  6557.7161  5.4351   False 2026-06-11
total_consumption_gwh         Naive with Drift   SARIMA   1.0.0  6709.3235  5.5662   False 2026-06-11
total_consumption_gwh             ARIMA(1,1,1)   SARIMA   1.0.0  6829.0903  5.6671   False 2026-06-11
total_consumption_gwh    SARIMAX(1,1,1) + Exog   SARIMA   1.0.0  9913.6518  8.2823   False 2026-06-11
total_consumption_gwh Random Forest Regression LightGBM   1.0.0 15957.4072 13.4064   False 2026-06-11


## Step 7: Register Peak Demand Model Separately

The ARIMA notebook also trained a separate model for `total_peak_demand_mw`. We register it as a distinct target with its own metrics.

In [9]:
# Peak demand was evaluated with ARIMA(1,1,1) in the notebook
# We have the metrics from the ARIMA notebook output
peak_metrics = {
    'mae': 1019.85,
    'rmse': 1299.27,
    'mape': 5.61,
    'test_period': '2021-2024',
    'train_period': '2003-2020',
    'n_train': 18,
    'n_test': 4
}

peak_model = {
    'model_name': 'ARIMA(1,1,1) — total_peak_demand_mw',
    'model_version': '1.0.0',
    'model_type': 'SARIMA',
    'target_variable': 'total_peak_demand_mw',
    'train_date': date.today().isoformat(),
    'metrics': peak_metrics,
    'model_path': 'forecast_peak_demand_2025_2030.csv',
    'is_active': False
}

peak_record = upsert_model(peak_model)
print(f"Inserted peak demand model: {peak_record['model_name']} ({peak_record['model_id']})")

# Activate it as the only model for this target
activated_peak = activate_model(peak_record['model_id'], 'total_peak_demand_mw')
print(f"Activated: {activated_peak['model_name']} | MAPE={activated_peak['metrics']['mape']:.2f}%")

Inserted peak demand model: ARIMA(1,1,1) — total_peak_demand_mw (025ac8d4-4968-4e11-9110-0afea52725ac)
Activated: ARIMA(1,1,1) — total_peak_demand_mw | MAPE=5.61%


## Step 8: Backend Integration Pattern

This is how the LUMI FastAPI backend can query the registry at runtime to serve the correct forecast:

In [10]:
def get_active_model_for_target(target: str) -> dict | None:
    """
    Fetch the active model record for a given target variable.
    This mirrors what the EnergyHub service would do at runtime.
    """
    resp = (
        supabase
        .table('ml_model_registry')
        .select('*')
        .eq('target_variable', target)
        .eq('is_active', True)
        .single()
        .execute()
    )
    return resp.data

# Example usage
for target in ['total_consumption_gwh', 'total_peak_demand_mw']:
    model = get_active_model_for_target(target)
    if model:
        print(f"\nActive for {target}:")
        print(f"  Name:    {model['model_name']}")
        print(f"  Type:    {model['model_type']}")
        print(f"  Version: {model['model_version']}")
        print(f"  Path:    {model['model_path']}")
        print(f"  MAPE:    {model['metrics']['mape']:.2f}%")
    else:
        print(f'No active model for {target}')


Active for total_consumption_gwh:
  Name:    Linear Trend Regression — total_consumption_gwh
  Type:    LightGBM
  Version: 1.0.0
  Path:    forecast_consumption_2025_2030.csv
  MAPE:    4.97%

Active for total_peak_demand_mw:
  Name:    ARIMA(1,1,1) — total_peak_demand_mw
  Type:    SARIMA
  Version: 1.0.0
  Path:    forecast_peak_demand_2025_2030.csv
  MAPE:    5.61%


## Step 9: Summary

| Target | Active Model | Type | MAPE | Test Period |
|--------|-------------|------|------|-------------|
| `total_consumption_gwh` | Linear Trend Regression | SARIMA* | 4.97% | 2021–2024 |
| `total_peak_demand_mw` | ARIMA(1,1,1) | SARIMA | 5.61% | 2021–2024 |


* *In our registry taxonomy, `SARIMA` covers ARIMA, SARIMAX, and Holt's smoothing since they are all Box-Jenkins / state-space statistical models. The `LightGBM` bucket covers Random Forest and Linear Regression as non-statistical baselines.*

### Registry Benefits
1. **Reproducibility:** Every model version is tracked with metrics and training date.
2. **A/B Testing:** New models can be registered and compared before activation.
3. **Rollback:** If a new model underperforms, deactivate it and reactivate the previous best.
4. **Dashboard Integration:** The EnergyHub backend queries `is_active=true` to serve the current forecast.
5. **Audit Trail:** The `created_at` / `updated_at` timestamps and `trg_ml_model_registry_updated` trigger maintain a history.